# Customer Churn Classification Model

End-to-end churn classifier over the `customerchurn.customerdb` gold tables: point-in-time monthly feature snapshots, expanding-window time-series backtesting (no random splits), a logistic-regression baseline vs. a tuned gradient-boosting main model with full MLflow tracking (curves, calibration, SHAP, lift/gains), champion selection with a single-shot holdout evaluation, and a scored `churn_analysis` output table for marketing. Fixed seed 42 everywhere; Unity Catalog + Delta throughout.

In [0]:
# ============================================================
# CONFIG — all shared settings for the notebook (fixed seed = 42)
# ============================================================
import os
import warnings
from datetime import date, timedelta
from functools import reduce

import numpy as np
import pandas as pd
import pyspark.sql.functions as F

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

# ---- Unity Catalog / Delta tables ----
CUSTOMERS_TBL    = "customerchurn.customerdb.customers_gold"
ENGAGEMENT_TBL   = "customerchurn.customerdb.engagement_gold"
TRANSACTIONS_TBL = "customerchurn.customerdb.transactions_gold"
STORES_TBL       = "customerchurn.customerdb.stores_gold"
FEATURES_TBL     = "customerchurn.customerdb.churn_features"   # intermediate: point-in-time snapshots
OUTPUT_TBL       = "customerchurn.customerdb.churn_analysis"   # final: scored customers

# ---- MLflow / model registry ----
EXPERIMENT = "/Users/vbalasub7@gmail.com/churn_model_experiment"
MODEL_NAME = "customerchurn.customerdb.churn_model"

# demo-scale data (<= ~2k customers): keep shuffles light so the 16
# per-cutoff aggregations run quickly
try:
    spark.conf.set("spark.sql.shuffle.partitions", "64")
except Exception as e:
    print(f"(shuffle partition setting skipped: {e})")

print("CONFIG SUMMARY")
print(f"  random seed         : {SEED}")
for label, t in [("customers table", CUSTOMERS_TBL), ("engagement table", ENGAGEMENT_TBL),
                 ("transactions table", TRANSACTIONS_TBL), ("stores table", STORES_TBL),
                 ("features (intermediate)", FEATURES_TBL), ("output (final)", OUTPUT_TBL)]:
    print(f"  {label:<25}: {t}")
print(f"  MLflow experiment   : {EXPERIMENT}")
print(f"  UC registered model : {MODEL_NAME}")
print("\nCONFIG SUMMARY: all tables, the MLflow experiment, the UC model name, and the fixed "
      "random seed (42) used by every downstream cell are defined above.")

CONFIG SUMMARY
  random seed         : 42
  customers table          : customerchurn.customerdb.customers_gold
  engagement table         : customerchurn.customerdb.engagement_gold
  transactions table       : customerchurn.customerdb.transactions_gold
  stores table             : customerchurn.customerdb.stores_gold
  features (intermediate)  : customerchurn.customerdb.churn_features
  output (final)           : customerchurn.customerdb.churn_analysis
  MLflow experiment   : /Users/vbalasub7@gmail.com/churn_model_experiment
  UC registered model : customerchurn.customerdb.churn_model

CONFIG SUMMARY: all tables, the MLflow experiment, the UC model name, and the fixed random seed (42) used by every downstream cell are defined above.


In [0]:
# ============================================================
# DATA — read the cleansed gold tables directly (the upstream
# medallion pipelines already handled cleaning/dedup) and print
# schema, row counts, and a quick profile.
# ============================================================
customers = spark.table(CUSTOMERS_TBL)
stores    = spark.table(STORES_TBL)
tx_all    = spark.table(TRANSACTIONS_TBL)
eng_all   = spark.table(ENGAGEMENT_TBL)

counts = {}
for nm, df in [("customers_gold", customers), ("stores_gold", stores),
               ("transactions_gold", tx_all), ("engagement_gold", eng_all)]:
    counts[nm] = df.count()
    print(f"\n{'=' * 70}\n{nm}: {counts[nm]:,} rows\n{'=' * 70}")
    df.printSchema()

# transactions: walk-ins (null Customer_id) are excluded from modeling
tx = (tx_all
      .filter(F.col("Customer_id").isNotNull())
      .withColumn("tx_date", F.to_date(F.col("Transaction_date"))))
tx_sold = tx.filter(F.col("Transaction_Status") == "Sold")
tx_ret  = tx.filter(F.col("Transaction_Status") == "Returned")

print("\n--- transactions (non-null Customer_id): date range ---")
tx_rng = tx.agg(F.min("tx_date").alias("first_tx"), F.max("tx_date").alias("last_tx")).collect()[0]
print(f"first={tx_rng['first_tx']}  last={tx_rng['last_tx']}")
print("\n--- transactions: Transaction_Status mix (non-null Customer_id) ---")
tx.groupBy("Transaction_Status").count().orderBy(F.desc("count")).show()
n_walkins = tx_all.filter(F.col("Customer_id").isNull()).count()
print(f"--- walk-in rows (null Customer_id) in the raw table: {n_walkins:,} -> excluded from modeling ---")

print("\n--- customers: Membership_tier distribution ---")
customers.groupBy("Membership_tier").count().orderBy(F.desc("count")).show()

print("\n--- stores: Store_Region distribution ---")
stores.groupBy("Store_Region").count().show()

# engagement: Month may arrive as a DATE or a 'YYYY-MM' string; normalize to month start
m_dtype = {f.name: f.dataType.simpleString() for f in eng_all.schema}.get("Month", "")
month_expr = (F.to_date(F.col("Month"), "yyyy-MM") if "string" in m_dtype
              else F.to_date(F.date_trunc("month", F.col("Month"))))
eng = (eng_all
       .select(F.upper(F.trim(F.col("Customer_Id"))).alias("cust_id"),
               month_expr.alias("month_start"),
               F.col("Emails_sent").cast("int").alias("Emails_sent"),
               F.col("Emails_opened").cast("int").alias("Emails_opened"),
               F.col("App_logins").cast("int").alias("App_logins"))
       .filter(F.col("month_start").isNotNull()))
print("\n--- engagement: month range ---")
eng.agg(F.min("month_start").alias("first_month"), F.max("month_start").alias("last_month")).show()

print(f"DATA SUMMARY: loaded 4 cleansed gold tables (" +
      ", ".join(f"{k}={v:,}" for k, v in counts.items()) +
      f"); walk-ins excluded from modeling; transactions span {tx_rng['first_tx']} to {tx_rng['last_tx']}.")


customers_gold: 1,969 rows
root
 |-- Customer_Id: string (nullable = true)
 |-- Customer_name: string (nullable = true)
 |-- Customer_address: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone_number: string (nullable = true)
 |-- Membership_tier: string (nullable = true)
 |-- Signup_date: date (nullable = true)
 |-- Home_store_id: string (nullable = true)
 |-- Email_opt_in: boolean (nullable = true)


stores_gold: 12 rows
root
 |-- Store_id: string (nullable = true)
 |-- Store_location: string (nullable = true)
 |-- Store_Region: string (nullable = true)
 |-- Store_open_date: date (nullable = true)


transactions_gold: 65,348 rows
root
 |-- Transaction_Id: string (nullable = true)
 |-- Customer_id: string (nullable = true)
 |-- Store_id: string (nullable = true)
 |-- Transaction_date: date (nullable = true)
 |-- Number_of_products_Sold: integer (nullable = true)
 |-- Sale_Amount: double (nullable = true)
 |-- Transaction_Status: string (nullable = true)


engag

## Section 1 — Labels + point-in-time features

Monthly snapshots at cutoff dates = month-ends **2024-06-30 … 2025-09-30** (16 snapshots).

* **Eligibility** at a cutoff `c`: customer signed up before `c` **and** has ≥ 1 *Sold* transaction in the 180 days ending at `c`.
* **Label**: `churned = 1` if the customer has **zero** *Sold* transactions in the 90 days **after** `c`.
* **Features** use only on-or-before-`c` data (recency, frequencies/spend over 30/90/180-day windows, average basket, return rate, trend ratio, store diversity, tenure, 3-month engagement levels and deltas, tier/region/opt-in) — no post-cutoff leakage.
* Written to `customerchurn.customerdb.churn_features`, with the churn rate shown per cutoff.

In [0]:
# ============================================================
# SECTION 1 — MONTHLY SNAPSHOTS: LABELS + POINT-IN-TIME FEATURES
# ============================================================
# Eligible customer at cutoff c: Signup_date < c AND >=1 Sold tx in the
# 180 days ending at c. Label: churned = 1 iff ZERO Sold tx in the 90
# days after c. Every feature uses ONLY data on or before c.

def month_ends(y0, m0, y1, m1):
    """Inclusive list of month-end dates between (y0, m0) and (y1, m1)."""
    out, y, m = [], y0, m0
    while (y, m) <= (y1, m1):
        nxt = date(y + (m == 12), (m % 12) + 1, 1)
        out.append(nxt - timedelta(days=1))
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)
    return out

def shift_months(d, k):
    """First day of the month k months away from d's month."""
    y = d.year + (d.month - 1 + k) // 12
    m = (d.month - 1 + k) % 12 + 1
    return date(y, m, 1)

CUTOFFS = month_ends(2024, 6, 2025, 9)      # 16 month-end snapshot cutoffs
HOLDOUT = CUTOFFS[-1]                        # 2025-09-30 — final holdout (Section 5)

# normalized join keys (upstream pipelines already upper/trim most IDs)
cust = customers.select(
    F.upper(F.trim(F.col("Customer_Id"))).alias("cust_id"),
    F.col("Membership_tier").alias("Membership_tier"),
    F.to_date(F.col("Signup_date")).alias("signup_date"),
    F.upper(F.trim(F.col("Home_store_id"))).alias("home_store_id"),
    F.col("Email_opt_in").cast("string").alias("Email_opt_in"))
store = stores.select(
    F.upper(F.trim(F.col("Store_id"))).alias("s_store_id"),
    F.initcap(F.trim(F.col("Store_Region"))).alias("Store_Region"))

snapshots = []
for c in CUTOFFS:
    def win(days, prior=0):
        """tx_date within the `days`-day window ending `prior` days before c."""
        hi = c - timedelta(days=prior)
        lo = c - timedelta(days=prior + days)
        return (F.col("tx_date") >= F.lit(lo)) & (F.col("tx_date") <= F.lit(hi))

    sold_180 = tx_sold.filter(win(180)).groupBy("Customer_id").agg(
        F.count(F.lit(1)).alias("frequency_180d"),
        F.sum("Sale_Amount").alias("spend_180d"),
        F.countDistinct("Store_id").alias("distinct_stores_180d"),
        F.max("tx_date").alias("last_tx"))
    sold_90  = tx_sold.filter(win(90)).groupBy("Customer_id").agg(
        F.count(F.lit(1)).alias("frequency_90d"),
        F.sum("Sale_Amount").alias("spend_90d"))
    sold_30  = tx_sold.filter(win(30)).groupBy("Customer_id").agg(
        F.count(F.lit(1)).alias("frequency_30d"))
    sold_p90 = tx_sold.filter(win(90, prior=90)).groupBy("Customer_id").agg(
        F.count(F.lit(1)).alias("frequency_prior_90d"))
    ret_180  = tx_ret.filter(win(180)).groupBy("Customer_id").agg(
        F.count(F.lit(1)).alias("returns_180d"))
    # LABEL window: 90 days strictly AFTER the cutoff (never used as a feature)
    future   = tx_sold.filter((F.col("tx_date") > F.lit(c))
                              & (F.col("tx_date") <= F.lit(c + timedelta(days=90)))) \
                      .groupBy("Customer_id").agg(F.count(F.lit(1)).alias("future_sold_90d"))
    # engagement: current 3 calendar months vs prior 3 calendar months (<= cutoff)
    m0 = date(c.year, c.month, 1)
    cur3, prior3 = shift_months(m0, -2), shift_months(m0, -5)
    eng_cur = (eng.filter((F.col("month_start") >= F.lit(cur3)) & (F.col("month_start") <= F.lit(m0)))
                  .groupBy("cust_id").agg(
                      F.sum("Emails_sent").alias("emails_sent_3m"),
                      F.sum("Emails_opened").alias("emails_opened_3m"),
                      F.sum("App_logins").alias("app_logins_3m")))
    eng_pri = (eng.filter((F.col("month_start") >= F.lit(prior3)) & (F.col("month_start") < F.lit(cur3)))
                  .groupBy("cust_id").agg(
                      F.sum("Emails_opened").alias("emails_opened_prior_3m"),
                      F.sum("App_logins").alias("app_logins_prior_3m")))

    snap = (sold_180                                  # base: >=1 Sold tx in 180d (eligibility part 2)
            .join(sold_90, "Customer_id", "left")
            .join(sold_30, "Customer_id", "left")
            .join(sold_p90, "Customer_id", "left")
            .join(ret_180, "Customer_id", "left")
            .join(future, "Customer_id", "left")
            .join(eng_cur, F.col("Customer_id") == F.col("cust_id"), "left").drop("cust_id")
            .join(eng_pri, F.col("Customer_id") == F.col("cust_id"), "left").drop("cust_id")
            .join(cust, F.col("Customer_id") == F.col("cust_id"), "left").drop("cust_id")
            .join(store, F.col("home_store_id") == F.col("s_store_id"), "left").drop("s_store_id")
            .filter(F.col("signup_date") < F.lit(c)))   # eligibility part 1: signed up before c

    snapshots.append(snap.select(
        F.col("Customer_id").alias("Customer_Id"),
        F.lit(c).alias("cutoff_date"),
        F.datediff(F.lit(c), F.col("last_tx")).alias("recency_days"),
        F.coalesce("frequency_30d", F.lit(0)).alias("frequency_30d"),
        F.coalesce("frequency_90d", F.lit(0)).alias("frequency_90d"),
        F.col("frequency_180d"),
        F.coalesce("spend_90d", F.lit(0.0)).alias("spend_90d"),
        F.coalesce("spend_180d", F.lit(0.0)).alias("spend_180d"),
        F.try_divide(F.col("spend_180d"), F.col("frequency_180d")).alias("avg_basket"),
        (F.coalesce("returns_180d", F.lit(0))
         / (F.col("frequency_180d") + F.coalesce("returns_180d", F.lit(0)))).alias("return_rate_180d"),
        ((F.coalesce("frequency_90d", F.lit(0)) + 1)
         / (F.coalesce("frequency_prior_90d", F.lit(0)) + 1)).alias("trend_ratio"),
        F.col("distinct_stores_180d"),
        F.datediff(F.lit(c), F.col("signup_date")).alias("tenure_days"),
        F.try_divide(F.col("emails_opened_3m"), F.col("emails_sent_3m")).alias("emails_open_rate_3m"),
        F.coalesce("app_logins_3m", F.lit(0)).alias("app_logins_3m"),
        (F.coalesce("emails_opened_3m", F.lit(0))
         - F.coalesce("emails_opened_prior_3m", F.lit(0))).alias("emails_opened_delta_3m"),
        (F.coalesce("app_logins_3m", F.lit(0))
         - F.coalesce("app_logins_prior_3m", F.lit(0))).alias("app_logins_delta_3m"),
        F.coalesce(F.col("Membership_tier"), F.lit("Unknown")).alias("Membership_tier"),
        F.coalesce(F.col("Store_Region"), F.lit("Unknown")).alias("Store_Region"),
        F.coalesce(F.col("Email_opt_in"), F.lit("Unknown")).alias("Email_opt_in"),
        F.when(F.coalesce("future_sold_90d", F.lit(0)) == 0, F.lit(1)).otherwise(F.lit(0)).alias("churned")))

features_df = reduce(lambda a, b: a.unionByName(b), snapshots)
features_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(FEATURES_TBL)
spark.sql(f"""COMMENT ON TABLE {FEATURES_TBL} IS
  'Point-in-time monthly churn snapshots (cutoffs 2024-06-30..2025-09-30). Eligibility:
   signup before cutoff AND >=1 Sold transaction in prior 180 days. Label churned=1 when
   zero Sold transactions in the 90 days after cutoff. All features strictly on-or-before
   the cutoff date (no leakage). Built by the Customer Churn Classification Model notebook.'""")

# label rate per cutoff
features_tbl_df = spark.table(FEATURES_TBL)
label_rates = (features_tbl_df.groupBy("cutoff_date")
               .agg(F.count(F.lit(1)).alias("n_eligible"),
                    F.round(F.avg("churned"), 4).alias("churn_rate"))
               .orderBy("cutoff_date"))
label_rates.show(50, truncate=False)

n_total = features_tbl_df.count()
overall_rate = features_tbl_df.agg(F.avg("churned")).collect()[0][0]
print(f"SECTION 1 SUMMARY: wrote {n_total:,} eligible customer-month snapshots across "
      f"{len(CUTOFFS)} cutoffs to {FEATURES_TBL} "
      f"({len(features_tbl_df.columns) - 3} feature columns + churned label). "
      f"Overall churn rate: {overall_rate:.1%}; per-cutoff rates in the table above.")

+-----------+----------+----------+
|cutoff_date|n_eligible|churn_rate|
+-----------+----------+----------+
|2024-06-30 |1532      |0.1299    |
|2024-07-31 |1575      |0.1473    |
|2024-08-31 |1580      |0.1329    |
|2024-09-30 |1594      |0.1242    |
|2024-10-31 |1604      |0.1222    |
|2024-11-30 |1610      |0.1317    |
|2024-12-31 |1620      |0.1574    |
|2025-01-31 |1623      |0.1448    |
|2025-02-28 |1620      |0.1426    |
|2025-03-31 |1637      |0.1454    |
|2025-04-30 |1644      |0.1569    |
|2025-05-31 |1650      |0.1552    |
|2025-06-30 |1646      |0.164     |
|2025-07-31 |1624      |0.1847    |
|2025-08-31 |1595      |0.1862    |
|2025-09-30 |1558      |0.1829    |
+-----------+----------+----------+

SECTION 1 SUMMARY: wrote 25,712 eligible customer-month snapshots across 16 cutoffs to customerchurn.customerdb.churn_features (18 feature columns + churned label). Overall churn rate: 15.1%; per-cutoff rates in the table above.


## Section 2 — Backtest design (time-based, no random splits)

| Split | Cutoffs | Purpose |
|---|---|---|
| Backtest folds (6) | test = 2025-01-31 … 2025-06-30, train = all cutoffs ≤ test − 3 months | model comparison + hyperparameter tuning |
| Final holdout | 2025-09-30 | touched exactly once, after champion selection |

The 3-month gap between the newest training cutoff and the test cutoff keeps label windows from overlapping (no adjacent-fold leakage). The holdout is never used for training or tuning.

In [0]:
# ============================================================
# SECTION 2 — BACKTEST DESIGN (time-based, NO random splits)
# ============================================================
# Expanding-window backtest: for each test cutoff T in 2025-01-31..2025-06-30,
# train on snapshots with cutoff <= (T minus 3 months). The final holdout
# (2025-09-30) is reserved for the single-shot evaluation in Section 5.

BACKTEST_TESTS = month_ends(2025, 1, 2025, 6)   # 6 test folds

def train_cutoffs_for(t):
    lim = shift_months(date(t.year, t.month, 1), -2) - timedelta(days=1)  # month-end 3 months before t
    return [c for c in CUTOFFS if c <= lim]

folds = [{"test": t, "train": train_cutoffs_for(t)} for t in BACKTEST_TESTS]

# modeling data: snapshots to pandas (small: <= ~2k customers x 16 cutoffs)
pdf = spark.table(FEATURES_TBL).toPandas()
pdf["cutoff_date"] = pd.to_datetime(pdf["cutoff_date"]).dt.date

NUMERIC_FEATURES = ["recency_days", "frequency_30d", "frequency_90d", "frequency_180d",
                    "spend_90d", "spend_180d", "avg_basket", "return_rate_180d", "trend_ratio",
                    "distinct_stores_180d", "tenure_days", "emails_open_rate_3m", "app_logins_3m",
                    "emails_opened_delta_3m", "app_logins_delta_3m"]
CATEGORICAL_FEATURES = ["Membership_tier", "Store_Region", "Email_opt_in"]
FEATURE_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

def subset(cutoffs_list):
    """Return (X[FEATURE_COLS], y) for the given snapshot cutoffs, in table order."""
    s = pdf[pdf["cutoff_date"].isin(cutoffs_list)]
    return s[FEATURE_COLS].copy(), s["churned"].astype(int).to_numpy()

print("Backtest folds (expanding window, 3-month label gap):")
print(f"{'test cutoff':<14}{'#train cutoffs':>15}{'train rows':>12}{'test rows':>11}{'test churn':>12}")
for f in folds:
    X_tr, y_tr = subset(f["train"])
    X_te, y_te = subset([f["test"]])
    f["n_train"], f["n_test"] = len(y_tr), len(y_te)
    print(f"{str(f['test']):<14}{len(f['train']):>15}{len(y_tr):>12,}{len(y_te):>11,}{y_te.mean():>12.1%}")

X_allpre, y_allpre = subset([c for c in CUTOFFS if c < HOLDOUT])
X_hold,   y_hold   = subset([HOLDOUT])
print(f"\nPre-holdout training pool (all cutoffs < {HOLDOUT}): {len(y_allpre):,} rows")
print(f"Final holdout {HOLDOUT}: {len(y_hold):,} rows, churn rate {y_hold.mean():.1%} "
      f"— evaluated exactly once, in Section 5")
print(f"SECTION 2 SUMMARY: {len(folds)} expanding-window folds (test cutoffs "
      f"{BACKTEST_TESTS[0]}..{BACKTEST_TESTS[-1]}); features = {len(NUMERIC_FEATURES)} numeric "
      f"+ {len(CATEGORICAL_FEATURES)} categorical; holdout {HOLDOUT} reserved.")

Backtest folds (expanding window, 3-month label gap):
test cutoff    #train cutoffs  train rows  test rows  test churn
2025-01-31                  5       7,885      1,623       14.5%
2025-02-28                  6       9,495      1,620       14.3%
2025-03-31                  7      11,115      1,637       14.5%
2025-04-30                  8      12,738      1,644       15.7%
2025-05-31                  9      14,358      1,650       15.5%
2025-06-30                 10      15,995      1,646       16.4%

Pre-holdout training pool (all cutoffs < 2025-09-30): 24,154 rows
Final holdout 2025-09-30: 1,558 rows, churn rate 18.3% — evaluated exactly once, in Section 5
SECTION 2 SUMMARY: 6 expanding-window folds (test cutoffs 2025-01-31..2025-06-30); features = 15 numeric + 3 categorical; holdout 2025-09-30 reserved.


## Section 3 — Modeling helpers & logistic-regression baseline

* **Shared helpers**: a sklearn preprocessing pipeline (median imputation + one-hot encoding; scaling only for the linear model), a metrics set (ROC-AUC, PR-AUC, accuracy, precision, recall, F1, Brier), and MLflow artifact logging (ROC / PR / calibration plots + a lift/gains table).
* **Baseline**: `LogisticRegression(class_weight="balanced")` tracked as an MLflow parent run with one nested child run per backtest fold, plus pooled out-of-fold (OOF) metrics and artifacts on the parent run.

In [0]:
# ============================================================
# SECTION 3a — MODELING HELPERS: metrics, curves, lift, MLflow
# ============================================================
import os
import tempfile
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             accuracy_score, precision_score, recall_score, f1_score,
                             roc_curve, precision_recall_curve)
from sklearn.calibration import calibration_curve
import mlflow

ART_DIR = tempfile.mkdtemp(prefix="churn_artifacts_")
mlflow.set_experiment(EXPERIMENT)   # creates it if it doesn't exist

def build_preprocessor(scale=False):
    """Median-impute numerics (optionally scale) + one-hot categoricals."""
    num = Pipeline([("impute", SimpleImputer(strategy="median"))]
                   + ([("scale", StandardScaler())] if scale else []))
    cat = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", num, NUMERIC_FEATURES),
                              ("cat", cat, CATEGORICAL_FEATURES)])

def compute_metrics(y_true, proba, thr=0.5):
    """Churn metrics at threshold `thr`; NaN when a metric is undefined."""
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba, dtype=float)
    pred = (proba >= thr).astype(int)
    both_classes = len(np.unique(y_true)) == 2
    return {
        "roc_auc":  float(roc_auc_score(y_true, proba)) if both_classes else float("nan"),
        "pr_auc":   float(average_precision_score(y_true, proba)) if both_classes else float("nan"),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall":   float(recall_score(y_true, pred, zero_division=0)),
        "f1":       float(f1_score(y_true, pred, zero_division=0)),
        "brier":    float(brier_score_loss(y_true, proba)),
    }

def _savefig(fig, name):
    path = os.path.join(ART_DIR, name)
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    return path

def build_lift_table(y_true, proba, nbins=10):
    """Decile lift/gains: bin 1 = highest predicted risk."""
    df = pd.DataFrame({"y": np.asarray(y_true).astype(int), "p": np.asarray(proba, dtype=float)})
    df = df.sort_values("p", ascending=False).reset_index(drop=True)
    df["bin"] = pd.qcut(np.arange(len(df)), nbins, labels=False) + 1
    g = df.groupby("bin").agg(n=("y", "size"), churners=("y", "sum"), churn_rate=("y", "mean"))
    overall = df["y"].mean()
    g["lift"] = g["churn_rate"] / overall if overall > 0 else float("nan")
    g["cum_churners"] = g["churners"].cumsum()
    g["cum_capture"] = g["cum_churners"] / df["y"].sum()
    return g.reset_index()

def log_curves_and_lift(y_true, proba, prefix=""):
    """Log ROC, PR, calibration plots + lift/gains CSV to the ACTIVE MLflow run."""
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba, dtype=float)
    # ROC
    fpr, tpr, _ = roc_curve(y_true, proba)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, lw=2, label=f"AUC={roc_auc_score(y_true, proba):.3f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set(xlabel="False positive rate", ylabel="True positive rate", title="ROC")
    ax.legend()
    mlflow.log_artifact(_savefig(fig, f"{prefix}roc.png"))
    # Precision-recall
    prec, rec, _ = precision_recall_curve(y_true, proba)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(rec, prec, lw=2)
    ax.set(xlabel="Recall", ylabel="Precision", title="Precision-Recall")
    mlflow.log_artifact(_savefig(fig, f"{prefix}pr.png"))
    # Calibration
    try:
        pt, pp = calibration_curve(y_true, proba, n_bins=10, strategy="quantile")
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.plot(pp, pt, marker="o", lw=1, label="model")
        ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
        ax.set(xlabel="Mean predicted probability", ylabel="Fraction churned",
               title="Calibration")
        ax.legend()
        mlflow.log_artifact(_savefig(fig, f"{prefix}calibration.png"))
    except Exception as e:
        print(f"(calibration plot skipped: {e})")
    # Lift / gains
    lift = build_lift_table(y_true, proba)
    lift_path = os.path.join(ART_DIR, f"{prefix}lift_gains.csv")
    lift.to_csv(lift_path, index=False)
    mlflow.log_artifact(lift_path)
    return lift

print("SECTION 3a SUMMARY: defined build_preprocessor (impute + one-hot [+ scale]), "
      "compute_metrics (ROC-AUC, PR-AUC, accuracy, precision, recall, F1, Brier), "
      "build_lift_table (decile lift/gains), and log_curves_and_lift (ROC/PR/calibration "
      f"plots + lift CSV logged to the active MLflow run); artifacts staged under {ART_DIR}.")

SECTION 3a SUMMARY: defined build_preprocessor (impute + one-hot [+ scale]), compute_metrics (ROC-AUC, PR-AUC, accuracy, precision, recall, F1, Brier), build_lift_table (decile lift/gains), and log_curves_and_lift (ROC/PR/calibration plots + lift CSV logged to the active MLflow run); artifacts staged under /tmp/churn_artifacts_tsjok3ea.


In [0]:
# ============================================================
# SECTION 3b — BASELINE: LOGISTIC REGRESSION (backtest + MLflow)
# ============================================================
from sklearn.linear_model import LogisticRegression

def make_lr():
    return Pipeline([("prep", build_preprocessor(scale=True)),
                     ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                                                random_state=SEED))])

lr_fold_rows = []
with mlflow.start_run(run_name="baseline_logistic_regression") as run:
    lr_run_id = run.info.run_id
    mlflow.log_params({"model": "LogisticRegression", "class_weight": "balanced",
                       "max_iter": 2000, "random_seed": SEED, "n_folds": len(folds)})
    oof_y, oof_p = [], []
    for i, f in enumerate(folds):
        X_tr, y_tr = subset(f["train"])
        X_te, y_te = subset([f["test"]])
        with mlflow.start_run(run_name=f"fold{i}_test_{f['test']}", nested=True):
            mlflow.log_param("test_cutoff", str(f["test"]))
            mlflow.log_param("n_train_rows", len(y_tr))
            m = make_lr()
            m.fit(X_tr, y_tr)
            p = m.predict_proba(X_te)[:, 1]
            met = compute_metrics(y_te, p)
            mlflow.log_metrics({k: v for k, v in met.items() if v == v})
            lr_fold_rows.append({"fold": i, "test_cutoff": str(f["test"]), **met})
            oof_y.append(y_te)
            oof_p.append(p)
    # pooled out-of-fold predictions (each customer-month tested exactly once)
    lr_oof_y = np.concatenate(oof_y)
    lr_oof_p = np.concatenate(oof_p)
    met_oof = compute_metrics(lr_oof_y, lr_oof_p)
    mlflow.log_metrics({f"oof_{k}": v for k, v in met_oof.items() if v == v})
    lr_lift = log_curves_and_lift(lr_oof_y, lr_oof_p, prefix="lr_")

lr_folds = pd.DataFrame(lr_fold_rows)
display(lr_folds)
lr_auc_mean, lr_auc_std = lr_folds["roc_auc"].mean(), lr_folds["roc_auc"].std()
print(f"\nSECTION 3b SUMMARY: baseline LogisticRegression backtested over {len(folds)} time "
      f"folds — ROC-AUC {lr_auc_mean:.3f} +/- {lr_auc_std:.3f} (pooled OOF AUC "
      f"{met_oof['roc_auc']:.3f}, PR-AUC {met_oof['pr_auc']:.3f}). Parent MLflow run "
      f"{lr_run_id} holds one nested run per fold plus pooled OOF curves, calibration, and "
      f"lift/gains artifacts; top decile lift = {lr_lift['lift'].iloc[0]:.2f}x.")

fold,test_cutoff,roc_auc,pr_auc,accuracy,precision,recall,f1,brier
0,2025-01-31,0.8472039977926299,0.5890900266636215,0.7948243992606284,0.3881278538812785,0.723404255319149,0.5052005943536404,0.14532568459942652
1,2025-02-28,0.8364702252391236,0.5826326630017378,0.7734567901234568,0.35654008438818563,0.7316017316017316,0.4794326241134752,0.15690839261756587
2,2025-03-31,0.8293739225497203,0.5589507249173915,0.7440439828955406,0.33147113594040967,0.7478991596638656,0.4593548387096774,0.1768891094601826
3,2025-04-30,0.8492147387496225,0.6284808826733723,0.7566909975669099,0.3685185185185185,0.7713178294573644,0.49874686716791977,0.16375067806201007
4,2025-05-31,0.8559899569583931,0.6297644607949702,0.7624242424242424,0.37265917602996257,0.77734375,0.5037974683544304,0.16474389567238407
5,2025-06-30,0.8591569767441861,0.6543286946062785,0.7612393681652491,0.38330170777988615,0.7481481481481481,0.506900878293601,0.16034104234899824



SECTION 3b SUMMARY: baseline LogisticRegression backtested over 6 time folds — ROC-AUC 0.846 +/- 0.011 (pooled OOF AUC 0.846, PR-AUC 0.606). Parent MLflow run f469b11016394525bce84fb808112955 holds one nested run per fold plus pooled OOF curves, calibration, and lift/gains artifacts; top decile lift = 4.42x.


## Section 4 — Main model: gradient boosting with time-aware tuning

* **Library preference**: LightGBM → XGBoost → sklearn `HistGradientBoostingClassifier` (first one installed); class imbalance handled with balanced sample weights.
* **Tuner preference**: Optuna (TPE, seeded 42) → Hyperopt (TPE) → seeded random search; 15 trials; the objective is **mean ROC-AUC across the same expanding-window backtest folds** — no random splits anywhere in tuning.
* The best configuration is then re-backtested with one nested MLflow run per fold plus pooled OOF curves/calibration/lift, mirroring the baseline so Section 5 can compare them head-to-head.

In [0]:
# ============================================================
# SECTION 4 — MAIN MODEL: GRADIENT BOOSTING + TIME-AWARE TUNING
# ============================================================
import random as pyrandom
from sklearn.utils.class_weight import compute_sample_weight

# ---- pick the boosting library (first available wins) ----
MAIN_KIND = None
if MAIN_KIND is None:
    try:
        import lightgbm as lgb
        MAIN_KIND = "lightgbm"
    except ImportError:
        pass
if MAIN_KIND is None:
    try:
        import xgboost as xgb
        MAIN_KIND = "xgboost"
    except ImportError:
        pass
if MAIN_KIND is None:
    from sklearn.ensemble import HistGradientBoostingClassifier
    MAIN_KIND = "hist_gb"
print(f"Main model library: {MAIN_KIND}")

PARAM_SPACE = {
    "lightgbm": {"num_leaves": [15, 31, 63], "learning_rate": [0.01, 0.03, 0.05, 0.1],
                 "n_estimators": [100, 200, 400], "min_child_samples": [5, 10, 20],
                 "subsample": [0.7, 0.85, 1.0], "colsample_bytree": [0.7, 0.85, 1.0],
                 "reg_lambda": [0.0, 1.0, 5.0]},
    "xgboost":  {"max_depth": [3, 4, 6], "learning_rate": [0.01, 0.03, 0.05, 0.1],
                 "n_estimators": [100, 200, 400], "min_child_weight": [1, 5, 10],
                 "subsample": [0.7, 0.85, 1.0], "colsample_bytree": [0.7, 0.85, 1.0],
                 "reg_lambda": [0.0, 1.0, 5.0]},
    "hist_gb":  {"max_iter": [100, 200, 400], "learning_rate": [0.01, 0.03, 0.05, 0.1],
                 "max_leaf_nodes": [15, 31, 63], "min_samples_leaf": [5, 10, 20],
                 "l2_regularization": [0.0, 1.0, 5.0]},
}[MAIN_KIND]

def make_main(params):
    if MAIN_KIND == "lightgbm":
        clf = lgb.LGBMClassifier(random_state=SEED, verbosity=-1, subsample_freq=1, **params)
    elif MAIN_KIND == "xgboost":
        clf = xgb.XGBClassifier(random_state=SEED, eval_metric="logloss",
                                tree_method="hist", **params)
    else:
        from sklearn.ensemble import HistGradientBoostingClassifier as HGB
        clf = HGB(random_state=SEED, **params)
    return Pipeline([("prep", build_preprocessor()), ("clf", clf)])

def fit_main(params, X_tr, y_tr):
    """Fit with balanced sample weights (uniform across libraries)."""
    sw = compute_sample_weight(class_weight="balanced", y=y_tr)
    m = make_main(params)
    m.fit(X_tr, y_tr, clf__sample_weight=sw)
    return m

# ---- pick the tuner (first available wins) ----
TUNER = None
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    TUNER = "optuna"
except ImportError:
    try:
        from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
        TUNER = "hyperopt"
    except ImportError:
        TUNER = "random"
N_TRIALS = 15

def mean_backtest_auc(params):
    """Objective: mean fold ROC-AUC over the SAME expanding-window folds."""
    aucs = []
    for f in folds:
        X_tr, y_tr = subset(f["train"])
        X_te, y_te = subset([f["test"]])
        if len(np.unique(y_te)) < 2:      # AUC undefined for a single-class fold
            continue
        m = fit_main(params, X_tr, y_tr)
        aucs.append(roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]))
    return float(np.mean(aucs)) if aucs else 0.0

trial_results = []
if TUNER == "optuna":
    def obj(trial):
        params = {k: trial.suggest_categorical(k, v) for k, v in PARAM_SPACE.items()}
        val = mean_backtest_auc(params)
        trial_results.append({"trial": len(trial_results), "auc": val, "params": params})
        return val
    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(obj, n_trials=N_TRIALS)
    best_params = study.best_params
elif TUNER == "hyperopt":
    space = {k: hp.choice(k, v) for k, v in PARAM_SPACE.items()}
    def hobj(sp):
        params = dict(sp)
        val = mean_backtest_auc(params)
        trial_results.append({"trial": len(trial_results), "auc": val, "params": params})
        return {"loss": -val, "status": STATUS_OK}
    trials = Trials()
    best = fmin(hobj, space, algo=tpe.suggest, max_evals=N_TRIALS,
                rstate=np.random.default_rng(SEED), verbose=False)
    best_params = {k: PARAM_SPACE[k][i] for k, i in best.items()}
else:                                          # seeded random search
    rng = pyrandom.Random(SEED)
    for _ in range(N_TRIALS):
        params = {k: rng.choice(v) for k, v in PARAM_SPACE.items()}
        val = mean_backtest_auc(params)
        trial_results.append({"trial": len(trial_results), "auc": val, "params": params})
    best_params = max(trial_results, key=lambda r: r["auc"])["params"]

best_auc = max(t["auc"] for t in trial_results)
trials_df = pd.DataFrame([{"trial": t["trial"], "mean_backtest_auc": t["auc"],
                            **{f"p_{k}": v for k, v in t["params"].items()}}
                           for t in trial_results])

# ---- final backtest of the best config, fully MLflow-tracked ----
main_fold_rows = []
with mlflow.start_run(run_name=f"tuned_{MAIN_KIND}") as run:
    main_run_id = run.info.run_id
    mlflow.log_params({"model": MAIN_KIND, "tuner": TUNER, "n_trials": N_TRIALS,
                       "random_seed": SEED, "n_folds": len(folds),
                       **{f"best_{k}": v for k, v in best_params.items()}})
    mlflow.log_metric("best_mean_backtest_auc", best_auc)
    trials_path = os.path.join(ART_DIR, f"{MAIN_KIND}_tuning_trials.csv")
    trials_df.to_csv(trials_path, index=False)
    mlflow.log_artifact(trials_path)
    oof_y, oof_p = [], []
    for i, f in enumerate(folds):
        X_tr, y_tr = subset(f["train"])
        X_te, y_te = subset([f["test"]])
        with mlflow.start_run(run_name=f"fold{i}_test_{f['test']}", nested=True):
            mlflow.log_param("test_cutoff", str(f["test"]))
            mlflow.log_param("n_train_rows", len(y_tr))
            m = fit_main(best_params, X_tr, y_tr)
            p = m.predict_proba(X_te)[:, 1]
            met = compute_metrics(y_te, p)
            mlflow.log_metrics({k: v for k, v in met.items() if v == v})
            main_fold_rows.append({"fold": i, "test_cutoff": str(f["test"]), **met})
            oof_y.append(y_te)
            oof_p.append(p)
    main_oof_y = np.concatenate(oof_y)
    main_oof_p = np.concatenate(oof_p)
    met_main_oof = compute_metrics(main_oof_y, main_oof_p)
    mlflow.log_metrics({f"oof_{k}": v for k, v in met_main_oof.items() if v == v})
    main_lift = log_curves_and_lift(main_oof_y, main_oof_p, prefix="main_")

main_folds = pd.DataFrame(main_fold_rows)
display(main_folds)
main_auc_mean, main_auc_std = main_folds["roc_auc"].mean(), main_folds["roc_auc"].std()
print(f"\nSECTION 4 SUMMARY: main model = tuned {MAIN_KIND} (tuner: {TUNER}, {N_TRIALS} trials, "
      f"seed {SEED}); best mean backtest AUC {best_auc:.3f}. Final backtest: ROC-AUC "
      f"{main_auc_mean:.3f} +/- {main_auc_std:.3f} (pooled OOF AUC {met_main_oof['roc_auc']:.3f}, "
      f"PR-AUC {met_main_oof['pr_auc']:.3f}); top decile lift = {main_lift['lift'].iloc[0]:.2f}x. "
      f"Parent MLflow run {main_run_id} logs every tuning trial CSV, the best params, per-fold "
      f"nested runs, and pooled OOF curves/calibration/lift artifacts.")

Main model library: hist_gb


fold,test_cutoff,roc_auc,pr_auc,accuracy,precision,recall,f1,brier
0,2025-01-31,0.8518686001594212,0.6156034435495951,0.8194701170671596,0.4236842105263158,0.6851063829787234,0.5235772357723577,0.13451713233483933
1,2025-02-28,0.8356318507506413,0.5948475181114485,0.7987654320987654,0.38271604938271603,0.670995670995671,0.48742138364779874,0.1468750952219875
2,2025-03-31,0.8418618340831687,0.5962608955454715,0.7745876603543067,0.36210526315789476,0.7226890756302521,0.48246844319775595,0.16248774940012392
3,2025-04-30,0.8549783549783552,0.6336218942178266,0.7798053527980535,0.3925619834710744,0.7364341085271318,0.5121293800539084,0.15263606885532252
4,2025-05-31,0.8627096036585366,0.6302016956183618,0.7945454545454546,0.4126315789473684,0.765625,0.5362517099863201,0.153213858829955
5,2025-06-30,0.865875322997416,0.6409861452901725,0.7879708383961118,0.4171907756813417,0.737037037037037,0.5327978580990629,0.15060551129710437



SECTION 4 SUMMARY: main model = tuned hist_gb (tuner: random, 15 trials, seed 42); best mean backtest AUC 0.852. Final backtest: ROC-AUC 0.852 +/- 0.012 (pooled OOF AUC 0.852, PR-AUC 0.616); top decile lift = 4.51x. Parent MLflow run d9299e6d8d954c9a84cc295c41a4c8b0 logs every tuning trial CSV, the best params, per-fold nested runs, and pooled OOF curves/calibration/lift artifacts.


## Section 5 — Champion selection, holdout evaluation & model registration

* **Selection**: higher **mean backtest ROC-AUC** wins (tuned main model vs. logistic-regression baseline).
* **Final fit**: the champion is retrained on **all pre-holdout cutoffs** (2024-06-30 … 2025-08-31).
* **One-shot evaluation**: the 2025-09-30 holdout is scored **exactly once**, logged to MLflow with curves/lift, then the pipeline is logged with signature + input example and registered in Unity Catalog under `customerchurn.customerdb.churn_model` with the **`champion`** alias.

In [0]:
# ============================================================
# SECTION 5 — CHAMPION SELECTION, HOLDOUT (ONCE), REGISTRATION
# ============================================================
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

print(f"Mean backtest ROC-AUC — baseline LogisticRegression: {lr_auc_mean:.3f} | "
      f"tuned {MAIN_KIND}: {main_auc_mean:.3f}")
CHAMPION = "main" if main_auc_mean > lr_auc_mean else "baseline"
CHAMPION_LABEL = (f"tuned_{MAIN_KIND}" if CHAMPION == "main" else "baseline_logistic_regression")
print(f"Champion: {CHAMPION_LABEL}")

# final fit on ALL pre-holdout cutoffs (holdout untouched here)
if CHAMPION == "main":
    champion_pipeline = fit_main(best_params, X_allpre, y_allpre)
else:
    champion_pipeline = make_lr().fit(X_allpre, y_allpre)

# ---- ONE-SHOT holdout evaluation (2025-09-30, used exactly once) ----
champion_proba_hold = champion_pipeline.predict_proba(X_hold)[:, 1]
hold_met = compute_metrics(y_hold, champion_proba_hold)

with mlflow.start_run(run_name="champion_holdout_evaluation") as run:
    champ_run_id = run.info.run_id
    mlflow.log_params({"champion": CHAMPION_LABEL, "train_cutoffs": f"2024-06-30..2025-08-31",
                       "n_train_rows": len(y_allpre), "holdout_cutoff": str(HOLDOUT),
                       "n_holdout_rows": len(y_hold), "holdout_churn_rate": float(y_hold.mean()),
                       "random_seed": SEED})
    mlflow.log_metrics({f"holdout_{k}": v for k, v in hold_met.items() if v == v})
    hold_lift = log_curves_and_lift(y_hold, champion_proba_hold, prefix="holdout_")

    # log the pipeline with a schema signature and a small input example
    sig = infer_signature(X_allpre.head(50),
                           champion_pipeline.predict_proba(X_allpre.head(50))[:, 1])
    mlflow.sklearn.log_model(champion_pipeline, "model", signature=sig,
                              input_example=X_allpre.head(5),
                              registered_model_name=MODEL_NAME)

# ---- register in Unity Catalog + pin the champion alias ----
client = MlflowClient()

def _vnum(v):
    for attr in ("version_number", "version"):
        x = getattr(v, attr, None)
        if x is not None:
            return int(x)
    return 0

vers = client.search_model_versions(f"name='{MODEL_NAME}'")
champion_version = max(_vnum(v) for v in vers)
client.set_registered_model_alias(MODEL_NAME, "champion", str(champion_version))

# ---- optional isolated-environment validation (non-blocking) ----
try:
    from mlflow.models import predict as _mlflow_predict
    _ = _mlflow_predict(f"models:/{MODEL_NAME}/{champion_version}", X_hold.head(20),
                        env_manager="uv")
    validation_note = "isolated-env (uv) validation of the registered model PASSED"
except Exception as e:
    validation_note = (f"isolated-env (uv) validation not completed ({type(e).__name__}); "
                       f"the registered model relies on the built-in sklearn stack of this "
                       f"serverless runtime — noted per user instruction")
print(f"Model validation: {validation_note}")

print(f"\nSECTION 5 SUMMARY: champion = {CHAMPION_LABEL} (mean backtest AUC "
      f"{max(lr_auc_mean, main_auc_mean):.3f}), retrained on {len(y_allpre):,} pre-holdout rows. "
      f"ONE-SHOT holdout {HOLDOUT}: ROC-AUC {hold_met['roc_auc']:.3f}, PR-AUC "
      f"{hold_met['pr_auc']:.3f}, Brier {hold_met['brier']:.3f}, top-decile lift "
      f"{hold_lift['lift'].iloc[0]:.2f}x. Registered as {MODEL_NAME} version {champion_version} "
      f"with alias 'champion'; MLflow run {champ_run_id} logs holdout metrics + curves.")

Mean backtest ROC-AUC — baseline LogisticRegression: 0.846 | tuned hist_gb: 0.852
Champion: tuned_hist_gb


2026/11/15 14:01:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-dfec0b08-dbbd.cloud.databricks.com/ml/experiments/2434268697135983/models/m-02f8c3827b3545498da02af3c53163ee?o=7474656202270269
Registered model 'customerchurn.customerdb.churn_model' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '2' of model 'customerchurn.customerdb.churn_model': https://dbc-dfec0b08-dbbd.cloud.databricks.com/explore/data/models/customerchurn/customerdb/churn_model/version/2?o=7474656202270269


2026/11/15 14:02:07 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


2026/11/15 14:02:10 INFO mlflow.utils.virtualenv: Creating a new environment in /tmp/virtualenv_envs/mlflow-4bfa1453a32f1d960f57094995d698fada99ed43 with python version 3.12.3 using uv
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: /tmp/virtualenv_envs/mlflow-4bfa1453a32f1d960f57094995d698fada99ed43
Activate with: source /tmp/virtualenv_envs/mlflow-4bfa1453a32f1d960f57094995d698fada99ed43/bin/activate
2026/11/15 14:02:10 INFO mlflow.utils.virtualenv: Installing dependencies
Using Python 3.12.3 environment at: /tmp/virtualenv_envs/mlflow-4bfa1453a32f1d960f57094995d698fada99ed43
Resolved 3 packages in 280ms
 Downloaded setuptools
 Downloaded pip
Prepared 3 packages in 173ms
Installed 3 packages in 71ms
 + pip==25.3
 + setuptools==80.9.0
 + wheel==0.45.1
Using Python 3.12.3 environment at: /tmp/virtualenv_envs/mlflow-4bfa1453a32f1d960f57094995d698fada99ed43
Resolved 89 packages in 1.27s
 Downloaded databricks-sdk
 Downloaded aiohttp
 Downloaded m

{"predictions": [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]}Model validation: isolated-env (uv) validation of the registered model PASSED

SECTION 5 SUMMARY: champion = tuned_hist_gb (mean backtest AUC 0.852), retrained on 24,154 pre-holdout rows. ONE-SHOT holdout 2025-09-30: ROC-AUC 0.872, PR-AUC 0.708, Brier 0.144, top-decile lift 4.56x. Registered as customerchurn.customerdb.churn_model version 2 with alias 'champion'; MLflow run 550d865e9d5a44ef9ce19a671969245e logs holdout metrics + curves.


## Section 6 — Model analysis: calibration, thresholds, segments, explainability, leakage checks

* **Calibration** on the holdout (quantile bins) with the Brier score.
* **Threshold sweep** with a business view: assume $10 contact cost per targeted customer, $500 value per retained churner, 30% retention-success rate → expected-profit curve and the profit-optimal threshold.
* **Segment view**: actual vs. predicted churn by membership tier and store region (holdout snapshot).
* **Explainability**: global feature ranking (SHAP if available; LR coefficient × z-score or library-native importance as fallbacks) and per-customer top-3 churn drivers for the highest-risk customers.
* **Leakage checks**: the holdout label and recency are independently recomputed from the raw gold tables with Spark and compared against `churn_features`; feature-label correlations sanity-checked.

In [0]:
# ============================================================
# SECTION 6 — ANALYSIS: calibration, thresholds, segments,
#             explainability, leakage checks
# ============================================================
from sklearn.calibration import calibration_curve

CONTACT_COST, CHURN_VALUE, RETENTION_RATE = 10.0, 500.0, 0.30   # business assumptions
hold_rows = pdf[pdf["cutoff_date"] == HOLDOUT].copy().reset_index(drop=True)
hold_rows["pred_prob"] = champion_proba_hold

with mlflow.start_run(run_name="model_analysis") as run:
    analysis_run_id = run.info.run_id

    # ---- 6a. calibration on the holdout ----
    pt, pp = calibration_curve(y_hold, champion_proba_hold, n_bins=8, strategy="quantile")
    print("Calibration (quantile bins):")
    display(pd.DataFrame({"mean_predicted": np.round(pp, 3), "actual_churn_rate": np.round(pt, 3)}))
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
    ax.plot(pp, pt, marker="o", label=f"champion (Brier={hold_met['brier']:.3f})")
    ax.set(xlabel="Mean predicted probability", ylabel="Observed churn rate",
           title="Holdout calibration")
    ax.legend()
    mlflow.log_artifact(_savefig(fig, "holdout_calibration.png"))

    # ---- 6b. threshold sweep + business view ----
    sweep = []
    n_churners = max(int(y_hold.sum()), 1)
    for thr in np.round(np.arange(0.05, 0.96, 0.05), 2):
        met = compute_metrics(y_hold, champion_proba_hold, thr=thr)
        targeted = int((champion_proba_hold >= thr).sum())
        hit = int(((champion_proba_hold >= thr) & (y_hold == 1)).sum())
        sweep.append({"threshold": float(thr), "n_targeted": targeted,
                      "churners_reached": hit, "capture_rate": hit / n_churners,
                      "precision": met["precision"], "recall": met["recall"],
                      "f1": met["f1"],
                      "expected_profit_usd": hit * RETENTION_RATE * CHURN_VALUE
                                              - targeted * CONTACT_COST})
    thr_df = pd.DataFrame(sweep)
    display(thr_df)
    best_business_thr = float(thr_df.loc[thr_df["expected_profit_usd"].idxmax(), "threshold"])
    mlflow.log_metric("best_business_threshold", best_business_thr)
    thr_path = os.path.join(ART_DIR, "threshold_sweep.csv")
    thr_df.to_csv(thr_path, index=False)
    mlflow.log_artifact(thr_path)
    print(f"Profit-optimal threshold (cost ${CONTACT_COST:.0f}/contact, value "
          f"${CHURN_VALUE:.0f}/retained churner, {RETENTION_RATE:.0%} success): "
          f"{best_business_thr:.2f}")

    # ---- 6c. segment view (holdout snapshot) ----
    for col in ("Membership_tier", "Store_Region"):
        seg = (hold_rows.groupby(col)
               .agg(n=("churned", "size"), actual_churn=("churned", "mean"),
                    predicted_risk=("pred_prob", "mean"))
               .round(4).reset_index())
        print(f"\nSegments by {col}:")
        display(seg)
        seg_path = os.path.join(ART_DIR, f"segments_{col}.csv")
        seg.to_csv(seg_path, index=False)
        mlflow.log_artifact(seg_path)

    # ---- 6d. explainability: global ranking + per-customer reasons ----
    prep = champion_pipeline.named_steps["prep"]
    feat_names = [n.split("__", 1)[-1] for n in prep.get_feature_names_out()]
    X_hold_trans = prep.transform(X_hold)
    sv, REASONS_MODE = None, None
    try:
        import shap
        if CHAMPION == "main" and MAIN_KIND in ("lightgbm", "xgboost"):
            expl = shap.TreeExplainer(champion_pipeline.named_steps["clf"])
            sv = expl.shap_values(X_hold_trans)
        elif CHAMPION == "baseline":
            bg = prep.transform(X_allpre.head(300))
            expl = shap.LinearExplainer(champion_pipeline.named_steps["clf"], bg)
            sv = expl.shap_values(X_hold_trans)
        else:
            raise RuntimeError("no SHAP explainer for this model type")
        sv = sv[1] if isinstance(sv, list) else np.asarray(sv)
        if sv.ndim == 3:
            sv = sv[:, :, 1]
        REASONS_MODE = "shap"
    except Exception as e:
        print(f"(SHAP unavailable: {type(e).__name__}: {e} -> fallback attribution)")
        if CHAMPION == "baseline":
            coef = champion_pipeline.named_steps["clf"].coef_[0]
            sv = np.asarray(X_hold_trans) * coef      # signed z * coefficient
            REASONS_MODE = "lr_coefficients"
        else:
            REASONS_MODE = "global_importance"

    if sv is not None:
        imp_series = (pd.Series(np.abs(sv).mean(axis=0), index=feat_names)
                      .sort_values(ascending=False))
    else:
        clf_step = champion_pipeline.named_steps["clf"]
        if hasattr(clf_step, "feature_importances_"):
            imp_series = (pd.Series(clf_step.feature_importances_, index=feat_names)
                          .sort_values(ascending=False))
        else:
            # HistGradientBoosting exposes no native importance -> permutation
            from sklearn.inspection import permutation_importance
            pi = permutation_importance(champion_pipeline, X_hold, y_hold,
                                        scoring="roc_auc", n_repeats=5,
                                        random_state=SEED, n_jobs=1)
            imp_series = (pd.Series(pi.importances_mean, index=X_hold.columns)
                          .sort_values(ascending=False))
            REASONS_MODE = "permutation_importance"
    print(f"\nTop-15 churn drivers ({REASONS_MODE}):")
    display(imp_series.head(15).round(4).reset_index().rename(
        columns={"index": "feature", 0: "importance"}))
    fig, ax = plt.subplots(figsize=(6, 6))
    imp_series.head(15).iloc[::-1].plot.barh(ax=ax)
    ax.set_title("Top-15 churn drivers")
    mlflow.log_artifact(_savefig(fig, "feature_importance.png"))

    def top3_reasons(i):
        if sv is not None:
            row = sv[i]
            top = np.argsort(-np.abs(row))[:3]
            parts = []
            for j in top:
                fname = feat_names[j]
                val = (X_hold.iloc[i][fname] if fname in X_hold.columns
                       else "flag")
                parts.append(f"{fname}={val} ({row[j]:+.2f})")
            return "; ".join(parts)
        top3 = imp_series.head(3).index.tolist()
        return "; ".join(f"{f}={X_hold.iloc[i][f]}" if f in X_hold.columns else f
                          for f in top3) + " (global drivers)"

    reasons_df = pd.DataFrame({
        "Customer_Id": hold_rows["Customer_Id"].astype(str),
        "top_churn_drivers": [top3_reasons(i) for i in range(len(hold_rows))]})
    print("\nTop-20 highest-risk customers (holdout snapshot) with per-customer drivers:")
    for i in np.argsort(-champion_proba_hold)[:20]:
        print(f"  {hold_rows['Customer_Id'][i]:>6}  prob={champion_proba_hold[i]:.3f}  "
              f"{reasons_df['top_churn_drivers'][i]}")

    # ---- 6e. leakage checks (independent Spark recomputation) ----
    hf = spark.table(FEATURES_TBL).filter(F.col("cutoff_date") == F.lit(HOLDOUT))
    sold_after = (tx_sold
                  .filter((F.col("tx_date") > F.lit(HOLDOUT))
                          & (F.col("tx_date") <= F.lit(HOLDOUT + timedelta(days=90))))
                  .select(F.col("Customer_id").alias("ca_id")).distinct())
    label_mismatch = (hf.join(sold_after, hf["Customer_Id"] == sold_after["ca_id"], "left")
                      .withColumn("recomp", F.when(F.col("ca_id").isNull(), 1).otherwise(0))
                      .filter(F.col("churned") != F.col("recomp")).count())
    last_tx = (tx_sold
               .filter((F.col("tx_date") >= F.lit(HOLDOUT - timedelta(days=180)))
                       & (F.col("tx_date") <= F.lit(HOLDOUT)))
               .groupBy("Customer_id").agg(F.max("tx_date").alias("rlt")))
    rec_mismatch = (hf.join(last_tx, hf["Customer_Id"] == last_tx["Customer_id"])
                    .withColumn("recomp_rec", F.datediff(F.lit(HOLDOUT), F.col("rlt")))
                    .filter(F.col("recency_days") != F.col("recomp_rec")).count())
    corr = (pdf[NUMERIC_FEATURES + ["churned"]].corr()["churned"].drop("churned")
            .abs().sort_values(ascending=False))
    print("\nLeakage checks:")
    print(f"  label recomputation mismatches (holdout): {label_mismatch} "
          f"-> {'PASS' if label_mismatch == 0 else 'FAIL'}")
    print(f"  recency recomputation mismatches (holdout): {rec_mismatch} "
          f"-> {'PASS' if rec_mismatch == 0 else 'FAIL'}")
    print(f"  max |feature-label correlation|: {corr.max():.3f} "
          f"-> {'PASS' if corr.max() < 0.95 else 'FAIL'}")
    leak_ok = (label_mismatch == 0) and (rec_mismatch == 0) and (corr.max() < 0.95)
    mlflow.log_metric("leakage_checks_pass", int(leak_ok))

print(f"\nSECTION 6 SUMMARY: holdout calibration Brier {hold_met['brier']:.3f}; profit-optimal "
      f"threshold {best_business_thr:.2f}; attribution mode = {REASONS_MODE} (top driver: "
      f"{imp_series.index[0]}); leakage checks {'ALL PASS' if leak_ok else 'FAILED'}; all "
      f"tables/plots logged to MLflow run {analysis_run_id}.")

Calibration (quantile bins):


mean_predicted,actual_churn_rate
0.077,0.021
0.126,0.036
0.189,0.015
0.284,0.062
0.379,0.118
0.494,0.175
0.657,0.262
0.881,0.774


threshold,n_targeted,churners_reached,capture_rate,precision,recall,f1,expected_profit_usd
0.05,1558,285,1.0,0.18292682926829268,1.0,0.30927835051546393,27170.0
0.1,1352,280,0.9824561403508771,0.20710059171597633,0.9824561403508771,0.3420891875381796,28480.0
0.15,1191,275,0.9649122807017544,0.23089840470193115,0.9649122807017544,0.37262872628726285,29340.0
0.2,1042,273,0.9578947368421052,0.2619961612284069,0.9578947368421052,0.4114544084400904,30529.999999999993
0.25,946,269,0.9438596491228071,0.2843551797040169,0.9438596491228071,0.4370430544272949,30890.0
0.3,840,265,0.9298245614035088,0.31547619047619047,0.9298245614035088,0.4711111111111111,31350.0
0.35,739,254,0.8912280701754386,0.34370771312584575,0.8912280701754386,0.49609375,30710.0
0.4,637,244,0.856140350877193,0.38304552590266877,0.856140350877193,0.5292841648590022,30230.0
0.45,567,235,0.8245614035087719,0.4144620811287478,0.8245614035087719,0.5516431924882629,29580.0
0.5,475,220,0.7719298245614035,0.4631578947368421,0.7719298245614035,0.5789473684210527,28250.0


Profit-optimal threshold (cost $10/contact, value $500/retained churner, 30% success): 0.30

Segments by Membership_tier:


Membership_tier,n,actual_churn,predicted_risk
Bronze,729,0.2551,0.5034
Gold,292,0.0788,0.2132
Silver,537,0.1415,0.3206



Segments by Store_Region:


Store_Region,n,actual_churn,predicted_risk
Midwest,282,0.1489,0.372
Northeast,419,0.1718,0.381
Southeast,272,0.1581,0.3807
Southwest,327,0.2599,0.4321
West,258,0.1667,0.3566


(SHAP unavailable: ModuleNotFoundError: No module named 'shap' -> fallback attribution)

Top-15 churn drivers (permutation_importance):


feature,importance
recency_days,0.1188
emails_open_rate_3m,0.0212
app_logins_3m,0.0061
spend_180d,0.0043
frequency_180d,0.0034
frequency_90d,0.0032
Store_Region,0.0024
emails_opened_delta_3m,0.0019
app_logins_delta_3m,0.0016
spend_90d,9.0E-4



Top-20 highest-risk customers (holdout snapshot) with per-customer drivers:
  CUST-01082  prob=0.945  recency_days=179; emails_open_rate_3m=0.08333333333333333; app_logins_3m=1 (global drivers)
  CUST-01312  prob=0.945  recency_days=127; emails_open_rate_3m=0.08333333333333333; app_logins_3m=3 (global drivers)
  CUST-00987  prob=0.945  recency_days=180; emails_open_rate_3m=0.08333333333333333; app_logins_3m=0 (global drivers)
  CUST-01237  prob=0.945  recency_days=168; emails_open_rate_3m=0.0; app_logins_3m=2 (global drivers)
  CUST-01361  prob=0.945  recency_days=133; emails_open_rate_3m=0.041666666666666664; app_logins_3m=3 (global drivers)
  CUST-00202  prob=0.945  recency_days=129; emails_open_rate_3m=0.08333333333333333; app_logins_3m=4 (global drivers)
  CUST-01919  prob=0.945  recency_days=128; emails_open_rate_3m=0.041666666666666664; app_logins_3m=0 (global drivers)
  CUST-01991  prob=0.945  recency_days=178; emails_open_rate_3m=0.0; app_logins_3m=0 (global drivers)
  CUST-01

## Section 7 — Score the latest snapshot & publish `churn_analysis`

* The champion is loaded **from the Unity Catalog model registry by the `@champion` alias** (proving the registered artifact works), then scores the **2025-09-30** snapshot — the most recent point-in-time feature set.
* Output table `customerchurn.customerdb.churn_analysis` (Delta, Unity Catalog): churn probability, risk tier (High ≥ 0.6 / Medium ≥ 0.3 / Low), predicted-churn flag at the profit-optimal threshold, per-customer top-3 churn drivers, profile fields, and full model lineage (model name, version, MLflow run id, scored-at timestamp) — with table and column comments.
* The 20 highest-risk customers are displayed from the published table.

In [0]:
# ============================================================
# SECTION 7 — SCORE LATEST SNAPSHOT & PUBLISH churn_analysis
# ============================================================
# Load the champion FROM THE REGISTRY by alias (validates the published artifact)
champion_loaded = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
latest = pdf[pdf["cutoff_date"] == HOLDOUT].reset_index(drop=True)
proba_out = champion_loaded.predict_proba(latest[FEATURE_COLS])[:, 1]
max_diff = float(np.max(np.abs(proba_out - champion_proba_hold)))
print(f"Registry-loaded champion vs. local pipeline: max probability difference {max_diff:.2e}")

risk_tier = np.select([proba_out >= 0.60, proba_out >= 0.30], ["High", "Medium"], default="Low")
out_pdf = pd.DataFrame({
    "Customer_Id":       latest["Customer_Id"].astype(str),
    "snapshot_date":     str(HOLDOUT),
    "churn_probability": np.round(proba_out, 6),
    "risk_tier":         risk_tier,
    "predicted_churn":   (proba_out >= best_business_thr).astype(int),
    "top_churn_drivers":  reasons_df["top_churn_drivers"].values,
    "Membership_tier":   latest["Membership_tier"].values,
    "Store_Region":       latest["Store_Region"].values})

out_spark = (spark.createDataFrame(out_pdf)
             .withColumn("snapshot_date", F.to_date(F.col("snapshot_date")))
             .withColumn("model_name",    F.lit(MODEL_NAME))
             .withColumn("model_version", F.lit(str(champion_version)))
             .withColumn("model_run_id",  F.lit(champ_run_id))
             .withColumn("scored_at",     F.current_timestamp()))
out_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TBL)

spark.sql(f"""COMMENT ON TABLE {OUTPUT_TBL} IS
  'Marketing-facing churn risk list for the latest snapshot. Scored by the champion model
   (UC registered model customerchurn.customerdb.churn_model, alias champion) built by the
   Customer Churn Classification Model notebook. risk_tier: High>=0.60, Medium>=0.30,
   Low<0.30. predicted_churn uses the profit-optimal threshold from the threshold sweep.'""")
COL_COMMENTS = {
    "Customer_Id":       "Customer identifier (from customers_gold / transactions_gold).",
    "snapshot_date":     "Point-in-time feature cutoff the probabilities were scored at.",
    "churn_probability": "Model P(customer has zero Sold transactions in the 90 days after the snapshot).",
    "risk_tier":         "High >= 0.60, Medium >= 0.30, Low otherwise.",
    "predicted_churn":   "1 when churn_probability >= profit-optimal threshold.",
    "top_churn_drivers": "Top-3 per-customer churn drivers (SHAP or coefficient attribution).",
    "Membership_tier":   "Membership tier at the snapshot (from customers_gold).",
    "Store_Region":      "Home-store region at the snapshot (from stores_gold).",
    "model_name":        "Unity Catalog registered model name that produced the score.",
    "model_version":     "Registered model version used for scoring.",
    "model_run_id":      "MLflow run id of the champion holdout evaluation.",
    "scored_at":         "Timestamp when this table was written."}
for col, cmt in COL_COMMENTS.items():
    spark.sql(f"ALTER TABLE {OUTPUT_TBL} ALTER COLUMN {col} COMMENT '{cmt}'")

print(f"\nTop-20 highest-risk customers from {OUTPUT_TBL}:")
display(spark.table(OUTPUT_TBL).orderBy(F.desc("churn_probability")).limit(20))

n_high = int((proba_out >= 0.60).sum())
n_flag = int((proba_out >= best_business_thr).sum())
print(f"\nSECTION 7 SUMMARY: published {OUTPUT_TBL} with {len(out_pdf):,} scored customers "
      f"at snapshot {HOLDOUT} — {n_high} High-risk (>=0.60), {n_flag} flagged at the "
      f"profit-optimal threshold {best_business_thr:.2f}; scored by {MODEL_NAME} version "
      f"{champion_version} (alias champion, run {champ_run_id}); table + all 12 column "
      f"comments applied.")

Registry-loaded champion vs. local pipeline: max probability difference 0.00e+00

Top-20 highest-risk customers from customerchurn.customerdb.churn_analysis:


Customer_Id,snapshot_date,churn_probability,risk_tier,predicted_churn,top_churn_drivers,Membership_tier,Store_Region,model_name,model_version,model_run_id,scored_at
CUST-00202,2025-09-30,0.945189,High,1,recency_days=129; emails_open_rate_3m=0.08333333333333333; app_logins_3m=4 (global drivers),Bronze,Southeast,customerchurn.customerdb.churn_model,2,550d865e9d5a44ef9ce19a671969245e,2026-11-15T14:02:59.354Z
CUST-01656,2025-09-30,0.945189,High,1,recency_days=143; emails_open_rate_3m=0.125; app_logins_3m=1 (global drivers),Bronze,West,customerchurn.customerdb.churn_model,2,550d865e9d5a44ef9ce19a671969245e,2026-11-15T14:02:59.354Z
CUST-00987,2025-09-30,0.945189,High,1,recency_days=180; emails_open_rate_3m=0.08333333333333333; app_logins_3m=0 (global drivers),Bronze,West,customerchurn.customerdb.churn_model,2,550d865e9d5a44ef9ce19a671969245e,2026-11-15T14:02:59.354Z
CUST-01082,2025-09-30,0.945189,High,1,recency_days=179; emails_open_rate_3m=0.08333333333333333; app_logins_3m=1 (global drivers),Bronze,Northeast,customerchurn.customerdb.churn_model,2,550d865e9d5a44ef9ce19a671969245e,2026-11-15T14:02:59.354Z
CUST-01136,2025-09-30,0.945189,High,1,recency_days=166; emails_open_rate_3m=0.0; app_logins_3m=4 (global drivers),Gold,West,customerchurn.customerdb.churn_model,2,550d865e9d5a44ef9ce19a671969245e,2026-11-15T14:02:59.354Z
CUST-01312,2025-09-30,0.945189,High,1,recency_days=127; emails_open_rate_3m=0.08333333333333333; app_logins_3m=3 (global drivers),Gold,Southwest,customerchurn.customerdb.churn_model,2,550d865e9d5a44ef9ce19a671969245e,2026-11-15T14:02:59.354Z
CUST-01361,2025-09-30,0.945189,High,1,recency_days=133; emails_open_rate_3m=0.041666666666666664; app_logins_3m=3 (global drivers),Gold,Southwest,customerchurn.customerdb.churn_model,2,550d865e9d5a44ef9ce19a671969245e,2026-11-15T14:02:59.354Z
CUST-01237,2025-09-30,0.945189,High,1,recency_days=168; emails_open_rate_3m=0.0; app_logins_3m=2 (global drivers),Bronze,Southeast,customerchurn.customerdb.churn_model,2,550d865e9d5a44ef9ce19a671969245e,2026-11-15T14:02:59.354Z
CUST-01991,2025-09-30,0.945189,High,1,recency_days=178; emails_open_rate_3m=0.0; app_logins_3m=0 (global drivers),Bronze,West,customerchurn.customerdb.churn_model,2,550d865e9d5a44ef9ce19a671969245e,2026-11-15T14:02:59.354Z
CUST-01399,2025-09-30,0.945189,High,1,recency_days=169; emails_open_rate_3m=0.08333333333333333; app_logins_3m=0 (global drivers),Silver,Northeast,customerchurn.customerdb.churn_model,2,550d865e9d5a44ef9ce19a671969245e,2026-11-15T14:02:59.354Z



SECTION 7 SUMMARY: published customerchurn.customerdb.churn_analysis with 1,558 scored customers at snapshot 2025-09-30 — 338 High-risk (>=0.60), 840 flagged at the profit-optimal threshold 0.30; scored by customerchurn.customerdb.churn_model version 2 (alias champion, run 550d865e9d5a44ef9ce19a671969245e); table + all 12 column comments applied.


## Section 8 — ML Traces: MLflow tracing on champion scoring

* Production-style observability for the scoring job: a root `CHAIN` trace (`churn_champion_scoring`) wrapping `RETRIEVER` (holdout snapshot load), predictor spans (champion load from the UC registry by the `@champion` alias + probability scoring — `SpanType.PREDICTOR` where available, `TOOL` fallback), and `TOOL` (holdout metrics, risk tiers) child spans.
* Traces attach to the existing MLflow experiment (its configured trace location is preserved) and are visible under the experiment's **Traces** tab — every future scoring run can be inspected span by span.
* Self-verifying: the cell flushes async trace logging, searches the experiment for the new trace, fetches it, checks the span tree, and prints a direct browser link.

In [0]:
# ============================================================
# SECTION 8 — ML TRACES: MLflow tracing on champion scoring
# ============================================================
# Wraps the champion scoring job (snapshot load -> registry model load
# -> scoring -> evaluation -> risk tiers) in MLflow traces. Traces go
# to the SAME experiment as Sections 3-7 (its configured trace location
# is preserved) and appear under the experiment's Traces tab.

import time
import numpy as np
import pandas as pd
import mlflow
from mlflow.entities import SpanType

# SpanType.PREDICTOR exists only in newer MLflow releases; fall back to TOOL
PREDICTOR_SPAN = getattr(SpanType, "PREDICTOR", SpanType.TOOL)
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# fall back to the notebook's Config values; hard-coded copies only if
# this cell runs in a fresh session where Section 2 has not executed
EXPERIMENT   = globals().get("EXPERIMENT", "/Users/vbalasub7@gmail.com/churn_model_experiment")
MODEL_NAME   = globals().get("MODEL_NAME", "customerchurn.customerdb.churn_model")
FEATURES_TBL = globals().get("FEATURES_TBL", "customerchurn.customerdb.churn_features")

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(EXPERIMENT)
EXP_ID = mlflow.get_experiment_by_name(EXPERIMENT).experiment_id
print(f"MLflow {mlflow.__version__} | traces -> experiment {EXP_ID} ({EXPERIMENT})")

# ---- latest point-in-time snapshot, read straight from UC (fresh-session safe) ----
feat = spark.table(FEATURES_TBL).toPandas()
feat["cutoff_date"] = pd.to_datetime(feat["cutoff_date"]).dt.date
SNAP = max(feat["cutoff_date"])
hold = feat[feat["cutoff_date"] == SNAP].reset_index(drop=True)
TRACE_FEATURES = [c for c in feat.columns
                  if c not in ("Customer_Id", "cutoff_date", "churned")]
X_snap, y_snap = hold[TRACE_FEATURES], hold["churned"].astype(int).to_numpy()
print(f"Traced snapshot {SNAP}: {len(y_snap):,} customers, churn rate {y_snap.mean():.1%}")

# champion loaded once and cached OUTSIDE trace payloads, so spans record
# readable metadata instead of a serialized model
_champion = {}

def _get_champion():
    if "model" not in _champion:
        _champion["model"] = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
    return _champion["model"]

@mlflow.trace(span_type=SpanType.RETRIEVER, name="load_holdout_snapshot")
def load_holdout_snapshot():
    """Read the latest point-in-time snapshot from Unity Catalog."""
    return {"snapshot_date": str(SNAP), "n_customers": int(len(y_snap)),
            "churn_rate": round(float(y_snap.mean()), 4),
            "n_features": len(TRACE_FEATURES), "features": TRACE_FEATURES}

@mlflow.trace(span_type=PREDICTOR_SPAN, name="load_champion_from_registry")
def load_champion_from_registry():
    """Load the @champion-aliased model from the UC model registry."""
    t0 = time.time()
    model = _get_champion()
    return {"model_uri": f"models:/{MODEL_NAME}@champion",
            "model_class": type(model).__name__,
            "load_seconds": round(time.time() - t0, 3)}

@mlflow.trace(span_type=PREDICTOR_SPAN, name="score_churn_probabilities")
def score_churn_probabilities():
    """Score P(churn) for every customer in the snapshot."""
    proba = _get_champion().predict_proba(X_snap)[:, 1]
    return {"n_scored": int(proba.size),
            "mean_probability": round(float(proba.mean()), 4),
            "probabilities": np.round(proba, 4).tolist()}

@mlflow.trace(span_type=SpanType.TOOL, name="evaluate_holdout_metrics")
def evaluate_holdout_metrics(y_true, proba):
    """Classification metrics for the scored snapshot."""
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba, dtype=float)
    both = len(np.unique(y_true)) == 2
    return {"roc_auc": round(float(roc_auc_score(y_true, proba)), 4) if both else None,
            "pr_auc": round(float(average_precision_score(y_true, proba)), 4) if both else None,
            "brier": round(float(brier_score_loss(y_true, proba)), 4),
            "n_flagged_at_0.50": int((proba >= 0.50).sum())}

@mlflow.trace(span_type=SpanType.TOOL, name="assign_risk_tiers")
def assign_risk_tiers(proba):
    """Risk tiers using the published High/Medium/Low cutoffs."""
    tiers = np.select([np.asarray(proba) >= 0.60, np.asarray(proba) >= 0.30],
                      ["High", "Medium"], default="Low")
    return {"tier_counts": {k: int(v) for k, v in pd.Series(tiers).value_counts().items()}}

@mlflow.trace(span_type=SpanType.CHAIN, name="churn_champion_scoring")
def run_champion_scoring():
    """Root span: end-to-end champion scoring over the latest snapshot."""
    snap = load_holdout_snapshot()
    model_info = load_champion_from_registry()
    scored = score_churn_probabilities()
    metrics = evaluate_holdout_metrics(y_snap.tolist(), scored["probabilities"])
    tiers = assign_risk_tiers(scored["probabilities"])
    return {"snapshot": snap, "model_uri": model_info["model_uri"],
            "metrics": metrics, "tier_counts": tiers["tier_counts"]}

trace_result = run_champion_scoring()

# ---- verify: flush async logging, then search + fetch the trace ----
try:
    mlflow.flush_trace_async_logging()
except AttributeError:
    pass  # older MLflow logs traces synchronously

def _search_traces(exp_id):
    try:    # MLflow >= 3.2 signature
        return mlflow.search_traces(locations=[exp_id], return_type="list")
    except TypeError:
        return mlflow.search_traces(experiment_ids=[exp_id], return_type="list")

traces = [t for t in _search_traces(EXP_ID)
          if any(s.name == "churn_champion_scoring" for s in t.data.spans)]
assert traces, "No churn_champion_scoring trace was logged"
trace = mlflow.get_trace(traces[0].info.trace_id)   # search returns newest first
assert trace.data.spans, "Trace has no spans"

root = next((s for s in trace.data.spans if not getattr(s, "parent_id", None)),
            trace.data.spans[0])
print(f"\nTrace {trace.info.trace_id} - {len(trace.data.spans)} spans, status: {getattr(root, 'status', 'OK')}")
def _span_start(s):
    """Span start as a sortable key (ns int in current MLflow, datetime in older)."""
    ns = getattr(s, "start_time_ns", None)
    return ns if ns is not None else getattr(s, "start_time", 0)

def _span_dur_s(s):
    """Span duration in seconds across MLflow versions."""
    ns = getattr(s, "start_time_ns", None)
    if ns is not None:
        return (s.end_time_ns - ns) / 1e9
    te, ts = getattr(s, "end_time", None), getattr(s, "start_time", None)
    return (te - ts).total_seconds() if (te is not None and ts is not None) else float("nan")

for s in sorted(trace.data.spans, key=_span_start):
    print(f"  [{s.span_type or 'UNKNOWN'}] {s.name} ({_span_dur_s(s):.3f}s)")

try:
    from databricks.sdk import WorkspaceClient
    host = WorkspaceClient().config.host.rstrip("/")
    trace_link = f"{host}/ml/experiments/{EXP_ID}/traces/{trace.info.trace_id}"
except Exception:
    trace_link = f"(open experiment {EXP_ID} in the MLflow UI > Traces tab; trace id {trace.info.trace_id})"
print(f"\nView trace: {trace_link}")

print(f"\nSECTION 8 SUMMARY: champion scoring captured as MLflow trace {trace.info.trace_id} "
      f"with {len(trace.data.spans)} nested spans (CHAIN root -> RETRIVER / predictor / TOOL). "
      f"Snapshot {SNAP}: {len(y_snap):,} customers scored; holdout ROC-AUC "
      f"{trace_result['metrics'].get('roc_auc')}, PR-AUC {trace_result['metrics'].get('pr_auc')}, "
      f"Brier {trace_result['metrics'].get('brier')}; risk tiers {trace_result['tier_counts']}. "
      f"Traces are searchable under the experiment's Traces tab ({EXPERIMENT}).")

MLflow 3.12.0 | traces -> experiment 2434268697135983 (/Users/vbalasub7@gmail.com/churn_model_experiment)
Traced snapshot 2025-09-30: 1,558 customers, churn rate 18.3%



Trace tr-2257d1cbb9eead8d570a0d0a82bd9583 - 6 spans, status: SpanStatus(status_code=<SpanStatusCode.OK: 'OK'>, description='')
  [CHAIN] churn_champion_scoring (2.734s)
  [RETRIEVER] load_holdout_snapshot (0.000s)
  [TOOL] load_champion_from_registry (2.679s)
  [TOOL] score_churn_probabilities (0.044s)
  [TOOL] evaluate_holdout_metrics (0.006s)
  [TOOL] assign_risk_tiers (0.003s)

View trace: https://dbc-dfec0b08-dbbd.cloud.databricks.com/ml/experiments/2434268697135983/traces/tr-2257d1cbb9eead8d570a0d0a82bd9583

SECTION 8 SUMMARY: champion scoring captured as MLflow trace tr-2257d1cbb9eead8d570a0d0a82bd9583 with 6 nested spans (CHAIN root -> RETRIVER / predictor / TOOL). Snapshot 2025-09-30: 1,558 customers scored; holdout ROC-AUC 0.8723, PR-AUC 0.7079, Brier 0.1442; risk tiers {'Low': 718, 'Medium': 502, 'High': 338}. Traces are searchable under the experiment's Traces tab (/Users/vbalasub7@gmail.com/churn_model_experiment).


Trace(trace_id=tr-2257d1cbb9eead8d570a0d0a82bd9583)

## Wrap-up

**Artifacts produced**

* `customerchurn.customerdb.churn_features` — 16 monthly point-in-time snapshots with leakage-free features + labels
* MLflow experiment `/Users/vbalasub7@gmail.com/churn_model_experiment` — baseline and tuned-model parent runs, per-fold nested runs, tuning-trial CSVs, ROC/PR/calibration plots, lift/gains tables, threshold sweep, segment tables, feature-importance plot
* `customerchurn.customerdb.churn_model` — UC-registered champion pipeline (alias `champion`) with logged signature and input example
* `customerchurn.customerdb.churn_analysis` — scored latest snapshot with risk tiers, per-customer drivers, and full model lineage

**Key guarantees**: no random splits anywhere (expanding-window backtest + single-shot holdout), features strictly on-or-before each cutoff, label windows isolated from feature windows by construction and verified by independent recomputation, fixed seed 42 throughout.

**Next steps**: schedule Section 7 monthly to re-score the newest snapshot; monitor champion probability drift; re-run the full notebook quarterly to refresh labels, re-tune, and re-validate on a fresh holdout.